# 公开 UPS 手册问答

我基于练习程序，把问答流程用于真实公开手册，跑了15个问题和取1块、3块的对比。这里保留原练习的实际输出，重新运行后会更新。原PDF和文字块在本地生成，来源和版本校验写在程序里。

## 运行准备

先在仓库根目录安装 `requirements.txt`，打开本Notebook，选择安装好依赖的Python内核。程序与问题集放在本文件夹。完整流程共17次DeepSeek请求，重跑会再次请求并更新结果文件。

## 提取手册

程序从官方来源下载PDF，选择7页，按260字切块，重叠60字。每块保留PDF页码和编号；表格中的列与数值仍需打开原页核对。下方路径输出来自原本地练习。

In [9]:
try:
    import manual_project as mp
except ModuleNotFoundError:
    from public_manual import manual_project as mp

records = mp.prepare_documents()
test_cases = mp.load_cases()
print("原PDF：", mp.BASE_DIR / "source_docs" / mp.MANUAL_NAME)
print("记录数量：", len(records))
print("测试问题数量：", len(test_cases))
print("第一条来源：", records[0]["source"])


原PDF： D:\1rance\LLM\LLM_07_公开手册问答项目\source_docs\EasyUPS_SRV_Chinese_990-91063B.pdf
记录数量： 26
测试问题数量： 15
第一条来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块01


## 编码原文并连接模型

原文只编码一次，每个问题复用 `doc_vectors`。如果修改记录，需要重新编码，保持文字、来源与向量顺序一致。密钥用隐藏输入。

In [3]:
from getpass import getpass
from openai import OpenAI

model = mp.load_search_model()
doc_vectors = mp.encode_documents(records, model)
print("原文向量形状：", doc_vectors.shape)

api_key = getpass("输入DeepSeek API key（隐藏输入）：").strip()
if not api_key:
    raise ValueError("请重新输入密钥")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com", timeout=60, max_retries=0)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

原文向量形状： (26, 512)


## 前五个问题

先取最相近的3块，记录候选原文、参考文字和实际回答。`status=ok`表示接口完成，`reference_page_hit`只表示候选命中了预期页，都不等于答案正确。

In [4]:
results = mp.run_cases(
    test_cases[:5], records, doc_vectors, model, client,
    top_k=3, output_name="llm07_first5.json",
)


R01 | 手册中输出电压的出厂默认设置是多少？ | ok
输出电压的出厂默认设置是 230 VAC。来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01]。

R02 | 输出电压允许选择哪些值？ | ok
输出电压允许选择 220、230、240 VAC。  
依据：“输出电压 230 VAC 220、230、240 VAC 允许用户在 UPS 在线操作时选择输出电压。”来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01。

R03 | 声音报警出厂时是开启还是关闭？ | ok
资料未提供。摘录仅提到“当实际运行时间达到用户设置的限制时，UPS 将发出声音警报”（PDF第15页，块03），并说明“仅当 UPS 在电池模式下工作时声音警报才会发出”，未说明声音报警的出厂默认开/关状态。

R04 | 绿色或高效率模式默认开启吗？ | ok
资料未提供。摘录中没有关于“绿色或高效率模式”默认开启状态的说明。

R05 | 输出电压的出厂默认设置是不是110V？ | ok
不是。输出电压的出厂默认设置是 230 VAC，用户可选择 220、230、240 VAC。  
来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01。



## 核对默认电压与错误前提

我对照PDF第15页审查了R01、R05：默认电压为230 VAC，110V的错误前提被纠正。下面保留本次判断；重跑后应根据新输出重新填写。

In [14]:
mp.inspect_result(results[0])
print("================")
mp.inspect_result(results[4])


# 以下是本次实际审查说明；重跑后需要重新核对。
results[0]["review"] = "通过：回答默认电压为230 VAC，与PDF第15页的默认设置一致"
results[4]["review"] = "通过：没有接受110V这个错误前提，纠正为230 VAC，引用与原文一致。"

编号： R01
问题： 手册中输出电压的出厂默认设置是多少？
预期： 230 VAC
状态： ok
来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块02 相似度： 0.604
电压处于
所配置输出电压的±5%范围内，和所
配置输出频率的±3 Hz 范围内，所连
设备即会通过旁路继电器接收市电输
入电源。在模式下逆变器关闭。
如果市电输入超出范围，逆变器会打
开。负载转移到在线模式或电池模
式。所连设备的供电可能中断最长 10
毫秒。
重新启动设置的
最小电池容量
0% 0%, 15%,50%,90%, 在电池的充电量能够提供此设置配置的
运行时间之前，UPS 输出不会打开。
如果配置为 0%，则在市电恢复后，
UPS 输出立即打开。
电池电量不足状
态指示设置
2 分钟 2 分钟、5 分

来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01 相似度： 0.564
UPS 设置
使用显示界面配置 UPS 设置。请参阅“配置 UPS 参数”章节以编辑参数。
功能 出厂默认设置 用户可选择的选项 说明
输出电压 230 VAC 220、230、240 VAC 允许用户在 UPS 在线操作时选择输出
电压。
声音报警 启用 启用、禁用 当设置为“禁用”或按下静音按钮时，
UPS 的声音警报将静音。
绿色模式/高效
率模式
禁用 启用/禁用 当此模式启用时，只要输入电压处于
所配置输出电压的±5%范围内，和所
配置输出频率的±3 Hz 范围内，所连
设备即会通过旁路继电器接收市电输

来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第6页，块02 相似度： 0.555
每个电池模块的电压 24 V 48 V 72 V
UPS 的总电压 24 V 48 V 72 V
Ah 额定值 每个电池模块 9 Ah
注释：
请参阅适用的电池更换用户手册，以了解安装说明。
请联系经销商或转到 APC by Schneider Electric 网站 www.apc.com 了解有关更换电池的信息。
前面板显示器
 UPS 电源开/关按钮
 警报 LED
 向上/向下按钮
 确认按钮
 静音/ESC 按钮
 LCD 显示屏
 状态指示灯

回答： 

## 取1块与3块的对比

对同一个PowerChute问题比较候选数量。原运行中两组都包含依据，回答和引用正确；参考文字从250字增加到872字，没有看到回答改善。单个问题的一次对比只作为观察。

In [ ]:
comparison = []
for top_k in [1, 3]:
    result = mp.answer_case(test_cases[5], records, doc_vectors, model, client, top_k=top_k)
    comparison.append(result)
    print("top_k：", top_k, "参考文字长度：", result["context_characters"])
    mp.inspect_result(result)
    print()

mp.save_json(comparison, "llm07_topk_comparison.json")

# 两组都检索到64位应用的依据。
# 两组回答和引用都符合原文。
# 文字从250字增加到872字，这个问题的回答没有改善。


top_k： 1 参考文字长度： 250
编号： R06
问题： 手册说PowerChute可以安装在32位系统上吗？
预期： 不可以；64位应用
状态： ok
来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03 相似度： 0.647
机。
注：PowerChute 是 64 位应用程序，无法安装
在 32 位操作系统上 。
3. 内置串行端口也可通过串行线缆用于附加通
讯选项。
注：RS232 和 USB 无法同时使用。
运行
使用显示屏
这些 Easy UPS 型号配备了直观和可配置的 LCD 显示屏。此显示屏对软件界面起到补充作用，可以传递类似信息，
也可用于配置 UPS 设置。
显示屏包含以下按键和指示灯:




回答： 不可以。手册说明：“PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03）

top_k： 3 参考文字长度： 872
编号： R06
问题： 手册说PowerChute可以安装在32位系统上吗？
预期： 不可以；64位应用
状态： ok
来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03 相似度： 0.647
机。
注：PowerChute 是 64 位应用程序，无法安装
在 32 位操作系统上 。
3. 内置串行端口也可通过串行线缆用于附加通
讯选项。
注：RS232 和 USB 无法同时使用。
运行
使用显示屏
这些 Easy UPS 型号配备了直观和可配置的 LCD 显示屏。此显示屏对软件界面起到补充作用，可以传递类似信息，
也可用于配置 UPS 设置。
显示屏包含以下按键和指示灯:




来源： EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块02 相似度： 0.639
详细信息，请登录 APC
网站(https://www.apc.com/us/en/faqs/home)查
看知识库中的 FAQ000223363 文章。
2. 对于带有操作系统的服务器或其他设备，请
登录 www.apc.com/pcss 下载并安装最新版
PowerChute Serial

WindowsPath('D:/1rance/LLM/LLM_07_公开手册问答项目/results/llm07_topk_comparison.json')

## 其余十个问题

保留全部15条结果，也保留失败。密码与当前电量两题检查资料缺失，静态手册不能提供这两种信息。

In [7]:
remaining = mp.run_cases(
    test_cases[5:], records, doc_vectors, model, client,
    top_k=3, output_name="llm07_remaining10.json",
)
all_results = results + remaining
mp.save_json(all_results, "llm07_all15.json")
print("完整测试结果数量：", len(all_results))
mp.inspect_result(all_results[-2])
mp.inspect_result(all_results[-1])


R06 | 手册说PowerChute可以安装在32位系统上吗？ | ok
不是。手册明确说明 PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上：“注：PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03；同页块02也有相同说明）

R07 | RS232和USB是否可以同时用于通信？ | ok
不可以。原句：“注：RS232 和 USB 无法同时使用。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03）

R08 | 手册给出的PowerChute软件下载网址是什么？ | ok
手册给出的下载网址是：`www.apc.com/pcss`。依据原文：“请登录 www.apc.com/pcss 下载并安装最新版PowerChute Serial Shutdown。”来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块02。

R09 | 事件日志最多记录最后多少个事件？ | ok
资料未提供。手册摘录只说明可按确认按钮查看日志、用向上/向下按钮查看记录的事件，以及日志末尾显示“End”，未给出最多记录多少个事件。[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第16页，块02]

R10 | 显示的剩余电池运行时间使用什么单位？ | ok
剩余电池运行时间使用“分钟”。手册第14页列出：“剩余电池运行时间 分钟 BAT, 分”。（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第14页，块02）

R11 | 输入频率范围是多少？ | ok
输入频率范围为 40–70 Hz。依据：“输入 电压 230 V AC 标称 频率 40 – 70 Hz”［EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第5页，块01］。

R12 | 手册列出的电池类型是什么？ | ok
手册列出的电池类型是“密封式免维护阀控式铅酸电池 (SMF-VRLA)”。（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第6页，块01）

R13 | 环境相对

## 本次观察

R03、R04、R09回答“资料未提供”，但原PDF有依据。候选都命中了预期页面，却漏掉了具体依据块：前两题需要第15页块01，日志题需要第16页块01。

我保留这些结果，用来区分“相关页面”“具体依据”和“正确回答”。本次是部分手册、固定问题的一次实验，没有自动评分。

[官方来源](https://www.se.com/tw/zh/download/document/SPD_AHUG-AS9C3R_CH/)，原文版权归厂商。